In [ ]:
import sys
sys.path.append('../src')
from should_be_stdlib import *
from data import *

In [ ]:
from itertools import combinations_with_replacement
from typing import Callable

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
from matplotlib import pyplot as plt
from seaborn import heatmap

In [ ]:
record = load_set()
tuning_curves = get_tc(record)
tuning_curves_rescaled = pd.read_csv(datapath('data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_resampled = pd.read_csv(datapath('data_tuning-curves_resampled.csv'), index_col=0)

In [ ]:
# pearson correlation
from scipy.stats import pearsonr

# euclidean distance
def euclidean(a, b):
    return np.linalg.norm(a - b)

# inverse (fast) fourier transform
from scipy.fft import ifft

# state fidelity
def fidelity(a, b):
    ra = a / np.linalg.norm(a, ord=1)
    rb = b / np.linalg.norm(a, ord=1)
    return np.sum(np.sqrt(ra * rb)) ** 2

In [ ]:
# metrics['metric'] = (data, calculator for rows)
metrics:dict[str,tuple[object,Callable[[list[float],list[float]],float]]] = {
    'correlation-pearson': (
        tuning_curves,
        lambda a, b: pearsonr(a, b).statistic
    ),
    'euclidean': (
        tuning_curves_rescaled,
        euclidean
    ),
    'euclidean-ifft': (
        tuning_curves_resampled,
        lambda a, b: euclidean(ifft(a), ifft(b))
    ),
    'classical-fidelity': (
        tuning_curves_rescaled,
        fidelity
    ),
    # can extend easily)
}

In [ ]:
def get_score(a_b):
    a, b = a_b
    return [a, b] + [
        v(d.loc[a], d.loc[b])
        for (_,(d,v)) in metrics.items()
    ]

In [ ]:
def get_scores():
    pairs = combinations_with_replacement(tuning_curves.index, 2)
    pairs_len = len(tuning_curves) * (len(tuning_curves) + 1) // 2

    from multiprocessing import Pool, cpu_count
    with Pool(processes=cpu_count()) as pool:
        ab = list(tqdm(pool.imap(get_score, pairs), total=pairs_len))

    return pd.DataFrame(
        ab,
        columns = ['A', 'B'] + [k for k in metrics.keys()]
    )

In [ ]:
csv = datapath('results_all_classical.csv')
if os.path.exists(csv):
    print('classical metrics already calculated')
    distances = pd.read_csv(csv)
else:
    distances = get_scores()
    distances.to_csv(csv)
distances = distances.pivot_table(index='A', columns='B').astype(np.float64)

In [ ]:
for metric in metrics.keys():
    distances[metric].to_csv(datapath(f'results_{metric}.csv'))
    plt.figure(figsize=(1,1), dpi=distances[metric].shape[0])
    ax = heatmap(mirror_matrix(distances[metric].to_numpy()), cbar=False, square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout(pad=0)
    plt.savefig(figspath(f'results_{metric}.png'))
    plt.show()
    plt.close()